# Maritime Video Inference

Runs the fine-tuned YOLO11s maritime detector (`best.pt`) on a video with identity-persistent, temporally-stable classification.

See `README.md` for full details on the smoothing/locking logic and configurable thresholds.

**Before running:** update `MODEL_PATH`, `VIDEO_PATH`, `OUTPUT_PATH`, and `TRACKER_CFG_PATH` below to match your local paths.

In [ ]:
import cv2
import yaml
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

# --- Configuration ---
MODEL_PATH = "./weights/best.pt"
VIDEO_PATH = "./input/video1.mp4"
OUTPUT_PATH = "./output/video1_smoothed.mp4"
TRACKER_CFG_PATH = "./bytetrack_stable.yaml"

# --- Smoothing / identity config ---
MIN_HITS_TO_DISPLAY = 5     # don't show a label until an identity has this many detections
SWITCH_MARGIN = 2.0         # new class's cumulative score must beat current by this factor to switch (only while unlocked)
DECAY = 0.98                # per-frame decay on old evidence (long memory, adapts very slowly)
LOCK_AFTER_HITS = 20        # once an identity has this many confirmed detections, PERMANENTLY freeze its class.
                             # Why: a real sustained model error (e.g. same ship called "tug" for 70+ straight
                             # frames due to a scale/angle change) will eventually out-vote any reactive smoothing
                             # no matter how decayed. Locking after enough confirmed hits trades "can self-correct
                             # forever" for "stays consistent for an object that's clearly already identified" --
                             # which is what you actually want for a single object tracked continuously in-frame.

# --- Re-link config (handles ByteTrack assigning a *new* ID to the same object) ---
RELINK_MAX_FRAMES_GONE = 45   # how many frames a track can be "lost" and still be re-linked
RELINK_MAX_CENTER_DIST = 80   # max center-point movement (px) to consider it the same object
RELINK_MAX_SIZE_RATIO = 1.6   # box area can't change by more than this factor between the two

# --- Box coasting (visual only): hold the last known box for a few frames if
# the detector/tracker drops the object for a moment, so it doesn't flicker
# in and out. Purely cosmetic -- does not affect classification evidence.
COAST_MAX_FRAMES = 8

# --- Write a custom ByteTrack config with a larger track buffer, so a
# briefly-occluded/missed vessel is far less likely to get dropped and
# reassigned a brand new ID in the first place. ---
tracker_cfg = {
    'tracker_type': 'bytetrack',
    'track_high_thresh': 0.25,
    'track_low_thresh': 0.1,
    'new_track_thresh': 0.4,
    'track_buffer': 60,       # default is 30 -- doubled to survive brief occlusions/misses
    'match_thresh': 0.85,
    'fuse_score': True,
}
with open(TRACKER_CFG_PATH, 'w') as f:
    yaml.safe_dump(tracker_cfg, f)

# Load the fine-tuned model
model = YOLO(MODEL_PATH)

# --- Video Writer Setup ---
cap = cv2.VideoCapture(VIDEO_PATH)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))


class Identity:
    """A persistent object identity. May outlive multiple raw tracker IDs
    if the tracker drops and reassigns a new ID for the same real object."""
    __slots__ = ("cls_scores", "committed_cls", "hits", "last_box",
                 "last_frame_seen", "raw_track_ids", "locked")

    def __init__(self):
        self.cls_scores = defaultdict(float)
        self.committed_cls = None
        self.hits = 0
        self.last_box = None          # (x1, y1, x2, y2)
        self.last_frame_seen = -1
        self.raw_track_ids = set()
        self.locked = False           # once True, committed_cls is frozen forever

    def update(self, cls_idx, conf, box, frame_idx):
        self.hits += 1
        self.last_box = box
        self.last_frame_seen = frame_idx

        if self.locked:
            return  # class already finalized -- ignore new evidence, keep coasting the box

        # Decay old evidence slightly before adding new evidence, so an
        # identity's classification is dominated by its long-run history
        # rather than the most recent few frames -- this is what stops a
        # correctly-classified "cargo" from drifting to "tug" after a
        # stretch of ambiguous frames.
        for k in self.cls_scores:
            self.cls_scores[k] *= DECAY
        self.cls_scores[cls_idx] += conf

        best_cls = max(self.cls_scores, key=self.cls_scores.get)
        if self.committed_cls is None:
            if self.hits >= MIN_HITS_TO_DISPLAY:
                self.committed_cls = best_cls
        elif best_cls != self.committed_cls:
            if self.cls_scores[best_cls] >= SWITCH_MARGIN * self.cls_scores.get(self.committed_cls, 1e-6):
                self.committed_cls = best_cls

        # Once we've seen enough confirmed detections, permanently lock the
        # class in. This is what actually guarantees consistency for an
        # object that stays in frame -- no amount of later disagreement
        # (even a long, confident stretch) can flip it after this point.
        if self.committed_cls is not None and self.hits >= LOCK_AFTER_HITS:
            self.locked = True


def box_center(b):
    x1, y1, x2, y2 = b
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def box_area(b):
    x1, y1, x2, y2 = b
    return max(0, x2 - x1) * max(0, y2 - y1)


# raw ByteTrack id -> Identity
identities = {}
# identities that stopped receiving updates recently (candidates for re-linking)
lost_identities = []  # list of Identity objects

# --- Inference Stream ---
# Using imgsz=640 to match training, tighter thresholds to stop ghost boxes,
# agnostic_nms to prevent overlapping predictions of different classes, and
# the custom tracker config above to reduce ID switches.
results = model.track(
    source=VIDEO_PATH,
    stream=True,
    imgsz=640,
    conf=0.40,
    iou=0.30,
    agnostic_nms=True,
    tracker=TRACKER_CFG_PATH,
    persist=True,
    device="cpu"           # set to 0 if you have a CUDA GPU available
)

print(f"Processing video. Output will be saved to: {OUTPUT_PATH}")

frame_idx = 0
seen_raw_ids_this_frame = set()

for r in results:
    frame_idx += 1
    frame = r.orig_img.copy()
    seen_raw_ids_this_frame.clear()

    if r.boxes is not None and r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy().astype(int)
        raw_ids = r.boxes.id.cpu().numpy().astype(int)
        clss = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        for box, raw_id, cls_idx, conf in zip(boxes, raw_ids, clss, confs):
            seen_raw_ids_this_frame.add(raw_id)

            identity = identities.get(raw_id)

            if identity is None:
                # New raw track ID from the tracker. Before treating it as a
                # brand-new object, check whether it's actually a recently
                # lost identity reappearing under a new ID (this is the main
                # fix for "cargo becomes tug after a while").
                best_match, best_dist = None, None
                for cand in lost_identities:
                    if frame_idx - cand.last_frame_seen > RELINK_MAX_FRAMES_GONE:
                        continue
                    if cand.last_box is None:
                        continue
                    dist = np.hypot(*(np.array(box_center(box)) - np.array(box_center(cand.last_box))))
                    if dist > RELINK_MAX_CENTER_DIST:
                        continue
                    area_ratio = max(box_area(box), 1) / max(box_area(cand.last_box), 1)
                    if not (1 / RELINK_MAX_SIZE_RATIO <= area_ratio <= RELINK_MAX_SIZE_RATIO):
                        continue
                    if best_dist is None or dist < best_dist:
                        best_match, best_dist = cand, dist

                if best_match is not None:
                    identity = best_match
                    lost_identities.remove(best_match)
                else:
                    identity = Identity()

                identity.raw_track_ids.add(raw_id)
                identities[raw_id] = identity

            identity.update(cls_idx, float(conf), tuple(box), frame_idx)

            if identity.committed_cls is None:
                continue  # not enough evidence yet -- don't draw a label

            class_name = model.names[identity.committed_cls]
            label_text = f"{class_name} {conf:.2f}"
            x1, y1, x2, y2 = box
            color = (0, 200, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
            cv2.putText(frame, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

    # Any raw ID that was active before but missing this frame becomes a
    # "lost" candidate for re-linking, and (briefly) coasts its last box so
    # the annotation doesn't flicker off for a couple of dropped frames.
    for raw_id, identity in list(identities.items()):
        if raw_id in seen_raw_ids_this_frame:
            continue
        gone_for = frame_idx - identity.last_frame_seen
        if gone_for == 1:
            lost_identities.append(identity)
        if gone_for <= COAST_MAX_FRAMES and identity.committed_cls is not None:
            class_name = model.names[identity.committed_cls]
            x1, y1, x2, y2 = identity.last_box
            color = (0, 150, 255)  # slightly different shade to indicate "coasting"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label_text = f"{class_name} (coasting)"
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
            cv2.putText(frame, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        elif gone_for > RELINK_MAX_FRAMES_GONE:
            # fully expire -- stop tracking/re-linking this identity
            identities.pop(raw_id, None)
            if identity in lost_identities:
                lost_identities.remove(identity)

    out.write(frame)

# Clean up
cap.release()
out.release()
print("Inference and rendering complete!")